# Add Image

## Imports

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

## The Machine Learning Pipeline in Action
It's time to build your first model! To solve the diabetes prediction problem, you will apply *relevant* stages of the **Machine Learning Pipeline**.

While the full pipeline provides a comprehensive framework, its strength is its adaptability. For this problem, you will focus on the essential steps needed to build a predictive model from your data.

### Stage 1 & 2: Data Ingestion and Preparation
Time to prepare your diabetes data for training. In the ML pipeline, this combines two stages: **Data Ingestion** (gathering raw data) and **Data Preparation** (cleaning it up). In more realistic projects, you'd pull patient records from a medical database and fix errors or missing values. For this project, that work is already done, but with a twist. This dataset contains medical measurements from multiple patients, which means your model will learn patterns between health indicators and diabetes diagnosis outcomes.

* Define the essential [tensors](https://docs.pytorch.org/docs/stable/tensors.html) for your task:
    * The **features** tensor contains patient health measurements including pregnancies, glucose levels, blood pressure, skin thickness, insulin, BMI, diabetes pedigree function, and age.
    * The **outcome** tensor shows whether each patient has diabetes (1) or not (0).
    * `dtype=torch.float32` sets your data type to 32-bit floating point values for precise calculations.

### Stage 1. Data Collection
**Dataset Overview:**
A medical dataset containing health measurements from female patients of Pima Indian heritage, used to predict diabetes diagnosis based on diagnostic measurements.

**Feature Descriptions:**

- **Pregnancies**: Number of times the patient has been pregnant
- **Glucose**: Plasma glucose concentration measured in a 2-hour oral glucose tolerance test (mg/dL)
- **BloodPressure**: Diastolic blood pressure measurement (mm Hg)
- **SkinThickness**: Triceps skin fold thickness measurement (mm)
- **Insulin**: 2-hour serum insulin level (mu U/ml)
- **BMI**: Body Mass Index - weight in kg/(height in m)²
- **DiabetesPedigreeFunction**: A function that scores likelihood of diabetes based on family history
- **Age**: Patient's age in years
- **Outcome**: Target variable - 1 indicates diabetes diagnosis, 0 indicates no diabetes

**Purpose**: Binary classification task to predict whether a patient has diabetes based on diagnostic measurements.

[Dataset link](https://www.kaggle.com/datasets/dohaesam/diabetes-dataset)

In [2]:
df = pd.read_csv('Data/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.shape

(768, 9)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


### Stage 2. Data Preparation

**i) Identifying and Marking Missing Values**
- Certain columns in this dataset use `0` to represent missing values, which is medically impossible (e.g., a person cannot have 0 blood pressure or 0 BMI)
- The code identifies five columns where `0` means "missing": Glucose, BloodPressure, SkinThickness, Insulin, and BMI
- Replaces all `0` values with `NaN` (Not a Number) to properly mark them as missing data

In [5]:
(df == 0).sum()

Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

In [6]:
# Columns where 0 means missing value
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Replace 0 with NaN
df[cols] = df[cols].replace(0, np.nan)

**ii) Imputing Missing Values**
- Fills all missing values (`NaN`) in the specified columns with the median value of that column
- **Why median?** It's more robust than mean for medical data, as it's not affected by extreme outliers
- This ensures every patient record has complete data for model training

In [7]:
# Fill missing values with median
df[cols] = df[cols].fillna(df[cols].median())

**iii) Statistical Summary**
- `df.describe()` generates summary statistics (count, mean, std, min, max, quartiles) for all numerical columns
- Helps verify the data preparation was successful and understand the distribution of values
- Useful for spotting any remaining anomalies or understanding the typical ranges of each feature

**Purpose**: These steps clean the dataset by handling missing values, ensuring the data is ready for Deep learning model training.

In [8]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,121.656250,72.386719,29.108073,140.671875,32.455208,0.471876,33.240885,0.348958
std,3.369578,30.438286,12.096642,8.791221,86.383060,6.875177,0.331329,11.760232,0.476951
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.078000,21.000000,0.000000
25%,1.000000,99.750000,64.000000,25.000000,121.500000,27.500000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,29.000000,125.000000,32.300000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


**iv) Separating Independent and Dependent Variables:**

In [9]:
# Independent variables
X = df.drop(columns=['Outcome'])

# Dependent variable
y = df['Outcome']

**v) Train-Test Split:**
- Divide data into training set (typically 70-80%) and testing set (20-30%)
- **Training set:** Used to train the model
- **Testing set:** Used to evaluate model performance on unseen data
- Prevents overfitting and gives realistic performance estimates

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2)

In [11]:
X_train

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
556,1,97.0,70.0,40.0,125.0,38.1,0.218,30
486,1,139.0,62.0,41.0,480.0,40.7,0.536,21
115,4,146.0,92.0,29.0,125.0,31.2,0.539,61
467,0,97.0,64.0,36.0,100.0,36.8,0.600,25
584,8,124.0,76.0,24.0,600.0,28.7,0.687,52
...,...,...,...,...,...,...,...,...
346,1,139.0,46.0,19.0,83.0,28.7,0.654,22
509,8,120.0,78.0,29.0,125.0,25.0,0.409,64
446,1,100.0,72.0,12.0,70.0,25.3,0.658,28
255,1,113.0,64.0,35.0,125.0,33.6,0.543,21


In [12]:
y

0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: Outcome, Length: 768, dtype: int64

### Stage 3: Feature Engineering

**Feature Scaling (Standardization/Normalization):**

- Transform features to a similar scale
- Common methods:
    - **StandardScaler:** Converts to mean=0, std=1 (standardization)
    - **MinMaxScaler:** Scales to range [0,1] (normalization)


**Why needed?** Features like Insulin (0-800) and Pregnancies (0-17) have different ranges. Many ML algorithms perform better when features are on similar scales
Apply scaling to training and testing sets separately to avoid data leakage

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
X_train

array([[-0.83211704, -0.82052036, -0.17149746, ...,  0.84362796,
        -0.78685477, -0.27147198],
       [-0.83211704,  0.55383525, -0.81899594, ...,  1.22377013,
         0.16279888, -1.03335831],
       [ 0.04776791,  0.78289452,  1.60912336, ..., -0.16521087,
         0.17175788,  2.35280315],
       ...,
       [-0.83211704, -0.7223521 , -0.00962284, ..., -1.02784118,
         0.52713141, -0.44078006],
       [-0.83211704, -0.29695632, -0.65712132, ...,  0.18568959,
         0.1837032 , -1.03335831],
       [ 0.63435788, -0.6569066 ,  1.44724874, ...,  0.4927275 ,
         0.57491273, -0.44078006]], shape=(614, 8))

In [15]:
y_train.values

array([0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0,
       1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,

**Converting to PyTorch Format:**

- `torch.from_numpy():` Converts NumPy arrays to PyTorch tensors
- Creates 4 tensors needed for training and evaluation:

    - `X_train_tensor:` Training features
    - `X_test_tensor:` Testing features
    - `y_train_tensor:` Training labels (outcomes)
    - `y_test_tensor:` Testing labels (outcomes)

In [16]:
X_train_tensor = torch.from_numpy(X_train.copy())
X_test_tensor = torch.from_numpy(X_test.copy())

y_train_tensor = torch.from_numpy(y_train.values.copy())
y_test_tensor = torch.from_numpy(y_test.values.copy())

In [17]:
X_train_tensor.shape

torch.Size([614, 8])

In [18]:
y_train_tensor.shape

torch.Size([614])

### Stage 4: Model Building

Now you'll create your model (this is stage 3 of the ML pipeline: Model Building). For diabetes prediction, you'll build a classification model that learns the relationship between health measurements and diabetes diagnosis. Your model will be a single-layer neural network (logistic regression) that learns these patterns.

A single neuron with multiple inputs implements a weighted combination followed by an activation function:

`P(Diabetes) = σ(W₁×Pregnancies + W₂×Glucose + W₃×BloodPressure + ... + W₈×Age + B)`

Where σ is the sigmoid function that converts the result into a probability between 0 and 1.

Your job is to find the best values for the weights (W₁, W₂, ..., W₈) and bias (B) that fit your patient data.

* **Custom Implementation**: The `MySimpleNN()` class creates a logistic regression model:
   * **Inputs**: 8 features (Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age)
   * **Output**: 1 probability value (likelihood of diabetes)
   * **Weights**: `self.weights` has shape (8, 1) - one weight per feature
   * **Bias**: `self.bias` starts at 0
   * **Activation**: Sigmoid function converts linear output to probability (0 to 1)
   * This single layer will learn which health indicators are most predictive of diabetes

* **Alternative with PyTorch `nn.Sequential`**:
   ```python
   model = nn.Sequential(
       nn.Linear(8, 1),    # 8 inputs → 1 output
       nn.Sigmoid()        # Convert to probability
   )
   ```
   * `nn.Linear(8, 1)`: Takes 8 health measurements as input, produces 1 output
   * `nn.Sigmoid()`: Squashes output to range [0, 1] for probability interpretation

In [19]:
class MySimpleNN():

  def __init__(self, X):

    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss

### Stage 5: Model Training

Time to train your neural network (this is stage 5 of the ML pipeline: Training). You need two key tools to help your model learn from the patient data:

**Loss Function: Binary Cross-Entropy (Custom Implementation)**
- Defined in `model.loss_function()`, it measures how wrong your probability predictions are
- For binary classification, it compares predicted probabilities to actual outcomes (0 or 1)
- If you predict 70% chance of diabetes but the patient doesn't have it (actual = 0), the loss function penalizes this error
- The model's goal is to minimize this loss, making predictions closer to actual diagnoses

**Optimizer: Manual Gradient Descent**
- Instead of using PyTorch's built-in optimizers like `optim.SGD` or `optim.Adam`, this implementation manually updates weights
- Uses the formula: `weight = weight - learning_rate × gradient`
- `learning_rate = 0.1`: This controls how big each adjustment step is
  - Too large (>0.5): Model might bounce around and never converge
  - Too small (<0.001): Training takes very long and might get stuck
  - 0.1 is relatively aggressive but works well for this simple model

**Training Process:**
- The model goes through **25 epochs** (complete passes through all training data)
- Each epoch: makes predictions → calculates loss → computes gradients → updates weights
- You'll see the loss printed after each epoch - watch it decrease as the model learns which health indicators predict diabetes!

**Note:** While this manual approach helps you understand the mechanics, PyTorch's built-in optimizers (`optim.SGD`, `optim.Adam`) are more efficient and offer advanced features like momentum and adaptive learning rates for production models.

#### Important Parameters

In [20]:
learning_rate = 0.1
epochs = 25

#### Training Pipeline

In [21]:
# create model
model = MySimpleNN(X_train_tensor)

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model.forward(X_train_tensor)

  # loss calculate
  loss = model.loss_function(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 1.352503625448018
Epoch: 2, Loss: 1.3237038422416718
Epoch: 3, Loss: 1.2953692415640226
Epoch: 4, Loss: 1.2675193550810773
Epoch: 5, Loss: 1.240174047859409
Epoch: 6, Loss: 1.2133534455615755
Epoch: 7, Loss: 1.187077851965309
Epoch: 8, Loss: 1.161367655371198
Epoch: 9, Loss: 1.1362432221429082
Epoch: 10, Loss: 1.11172477542075
Epoch: 11, Loss: 1.0878322570267007
Epoch: 12, Loss: 1.064585170790099
Epoch: 13, Loss: 1.0420024060071431
Epoch: 14, Loss: 1.0201020405248644
Epoch: 15, Loss: 0.9989011240118649
Epoch: 16, Loss: 0.9784154433205168
Epoch: 17, Loss: 0.9586592734076446
Epoch: 18, Loss: 0.9396451189798806
Epoch: 19, Loss: 0.921383453747434
Epoch: 20, Loss: 0.9038824657528514
Epoch: 21, Loss: 0.88714781850978
Epoch: 22, Loss: 0.8711824384520599
Epoch: 23, Loss: 0.8559863392849666
Epoch: 24, Loss: 0.841556493128369
Epoch: 25, Loss: 0.8278867568089371


### Make Your Prediction

In [25]:
# Use the torch.no_grad() context manager for efficient predictions
with torch.no_grad():
    # Define a new patient's health measurements
    # [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age]
    new_patient = pd.DataFrame({
        'Pregnancies': [2],
        'Glucose': [138],
        'BloodPressure': [62],
        'SkinThickness': [35],
        'Insulin': [0],
        'BMI': [33.6],
        'DiabetesPedigreeFunction': [0.127],
        'Age': [47]
    })
    
    # Convert the patient data into a 2D PyTorch tensor that the model expects
    # Note: Must apply the same scaling transformation used during training
    new_patient_scaled = scaler.transform(new_patient)  # Scale using the fitted scaler
    new_patient_tensor = torch.tensor(new_patient_scaled, dtype=torch.float64)
    
    # Pass the new data to the trained model to get a prediction
    diabetes_probability = model.forward(new_patient_tensor)
    
    # Use .item() to extract the scalar value from the tensor for printing
    prob_value = diabetes_probability.item()
    print(f"Diabetes Probability: {prob_value:.2%}")
    
    # Use the scalar value in a conditional statement to make the final decision
    # Threshold of 0.5 is standard for binary classification
    if prob_value > 0.5:
        print(f"\nPrediction: HIGH RISK - This patient is likely to have diabetes.")
        print(f"Recommendation: Further medical testing and consultation advised.")
    else:
        print(f"\nPrediction: LOW RISK - This patient is unlikely to have diabetes.")
        print(f"Recommendation: Continue routine health monitoring.")

    # Display patient profile
    print(f"\nPatient Profile:")
    for feature, value in new_patient.iloc[0].items():
        print(f"  {feature}: {value}")

Diabetes Probability: 51.57%

Prediction: HIGH RISK - This patient is likely to have diabetes.
Recommendation: Further medical testing and consultation advised.

Patient Profile:
  Pregnancies: 2.0
  Glucose: 138.0
  BloodPressure: 62.0
  SkinThickness: 35.0
  Insulin: 0.0
  BMI: 33.6
  DiabetesPedigreeFunction: 0.127
  Age: 47.0


### Inspecting the Model's Learning

Now that you have a working model, let's see the exact relationship it learned from the data. You can do this by inspecting the model's internal parameters, the final **weight** and **bias** values it discovered during training. These values define the precise line your model is now using to make predictions.

In [27]:
# Get the learned parameters
weights = model.weights.data.numpy()
bias = model.bias.data.numpy()

# Feature names
feature_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("="*60)
print("LEARNED MODEL PARAMETERS")
print("="*60)

print("\nFeature Weights (Importance):")
print("-" * 60)
for feature, weight in zip(feature_names, weights.flatten()):
    direction = "increases" if weight > 0 else "decreases"
    print(f"  {feature:25s}: {weight:8.4f}  ({direction} risk)")

print(f"\nBias (Baseline): {bias[0]:.4f}")
print("="*60)

# Show most important features
print("\nMost Important Features:")
weight_importance = [(name, abs(w)) for name, w in zip(feature_names, weights.flatten())]
weight_importance.sort(key=lambda x: x[1], reverse=True)

for i, (feature, importance) in enumerate(weight_importance[:3], 1):
    print(f"  {i}. {feature}: {importance:.4f}")

LEARNED MODEL PARAMETERS

Feature Weights (Importance):
------------------------------------------------------------
  Pregnancies              :   0.2427  (increases risk)
  Glucose                  :   0.3879  (increases risk)
  BloodPressure            :   0.3953  (increases risk)
  SkinThickness            :   0.3705  (increases risk)
  Insulin                  :  -0.1496  (decreases risk)
  BMI                      :   0.4618  (increases risk)
  DiabetesPedigreeFunction :   0.1178  (increases risk)
  Age                      :   0.1209  (increases risk)

Bias (Baseline): -0.3023

Most Important Features:
  1. BMI: 0.4618
  2. BloodPressure: 0.3953
  3. Glucose: 0.3879


### Evaluation

In [24]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.6497722864151001
